# COMPASS multivariate_longitudinal models

Dynamic-DeepHit runs both platinum and NEPC at every landmark, each with a cause-only and a cause-plus-death competing-risk configuration. The endpoints read their independent prediction-input trees created by `01_preprocessing.ipynb`.

Optional cluster paths are configured by endpoint in `PREDICTION_INPUT_DIRS_BY_ENDPOINT`. SurvLatent ODE remains off by default and uses the same four configurations when enabled.

In [ ]:
from pathlib import Path

ARMS = ["adt"]
ENDPOINTS = ("platinum", "nepc", "avpc")
COHORTS = ("all", "metastatic", "localized")
RUN_SURVLATENT = False
OVERWRITE = False  # True: refit and replace existing outputs; False: resume/skip

# Optional endpoint -> arm -> path overrides for GPU/shared-filesystem jobs.
PREDICTION_INPUT_DIRS_BY_ENDPOINT = {
    # "platinum": {"adt": Path("/cluster/path/to/prediction_inputs_adt")},
    # "nepc": {"adt": Path("/cluster/path/to/prediction_inputs_adt_nepc")},
}

import sys
sys.path.insert(0, ".")
import compass_pipeline as cp

cp.N_FOLDS = 5
cp.MAX_PRED_WINDOW = 260
cp.RUN_SURVLATENT = RUN_SURVLATENT
cp.FORCE_RERUN = OVERWRITE

if RUN_SURVLATENT:
    cp.SURVLATENT_REPO = cp.DEFAULT_SURVLATENT_REPO

RUNS = cp.make_endpoint_runs(
    ARMS, endpoints=ENDPOINTS, cohorts=COHORTS,
    prediction_input_dirs_by_endpoint=PREDICTION_INPUT_DIRS_BY_ENDPOINT,
)
for endpoint in ENDPOINTS:
    print(f"endpoint={endpoint}  configs={[c for _, c, _ in cp.longitudinal_task_specs(endpoint)]}")

## Run multivariate_longitudinal models

Dynamic-DeepHit in both of the endpoint's configs (and SurvLatent ODE too, if
`RUN_SURVLATENT` was set above). Set `OVERWRITE = True` in the
configuration cell to refit and replace existing outputs. With `False`, tasks
whose metrics file already exists are skipped. Layout:
`local_runs_<arm>[_nepc]/multivariate_longitudinal/<model>/landmark_{0,90,180}/<config>/`.

In [ ]:
for run in RUNS:
    cp.run_multivariate_longitudinal(run)

## Summary tables

Per-run C-index / mean AUC(t) / integrated Brier for every (model, landmark,
config), filtered to the endpoint's cause-of-interest row for headline
comparability against `03_multivariate.ipynb`'s Cox/XGBoost arms, then
combined across runs. The competing configs' death rows are written to disk
but excluded here — read them from the metrics CSVs directly if needed.

In [ ]:
summary_dfs = {cp.run_key(run): cp.summarize_longitudinal_outputs(run) for run in RUNS}
for label, df in summary_dfs.items():
    print(f"=== {label} ===")
    display(df)

In [ ]:
import pandas as pd

combined_longitudinal_summary_df = pd.concat(summary_dfs.values(), ignore_index=True)
combined_longitudinal_summary_df